## Gemini API를 활용한 제품 리뷰 비교 분석



**목표:**
1. 동일 품목(선크림)의 다양한 제품 리뷰 수집 및 분석
2. 제품별 장단점 자동 추출
3. 경쟁사 제품 비교 분석
4. 시각화를 통한 인사이트 도출
5. Gemini를 활용한 종합 인사이트 도출

#### 저장된 분석 파일 불러오기

In [ ]:
# =============================================================================
# 최신 제품 리뷰 분석 결과 불러오기
# =============================================================================

import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Dict, Optional, Any
import enum
from tqdm import tqdm
from datetime import datetime
from pprint import pprint
import time
import glob

# 현재 디렉토리에서 파일 패턴에 맞는 모든 JSON 파일 찾기
file_list = glob.glob("product_review_analysis_*.json")

if not file_list:
    print("분석 결과 파일이 존재하지 않습니다.")
else:
    # 최신 파일 기준: 수정 시간이 가장 늦은 파일
    latest_file = max(file_list, key=os.path.getmtime)
    
    print(f"최신 분석 파일 불러오기: {latest_file}")
    
    # JSON 파일을 DataFrame으로 직접 불러오기
    analysis_df = pd.read_json(latest_file, orient='records')
    
    print(f"불러온 리뷰 수: {len(analysis_df)}개")
    print(f"컬럼: {list(analysis_df.columns)}")
    
    print("\n데이터 샘플:")
    display(analysis_df.head(3))

 ### 1. 제품 비교 분석

In [ ]:
# 비교 분석 저장 폴더 생성
output_dir = "analysis_outputs"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
from collections import defaultdict, Counter

print("제품 비교 분석 시작...\n")

# 불용어 정의
stopwords = {'없음', '있음', '사용', '좋음', '나쁨', '보통', '괜찮음', '추천', '적극 추천', '만족', '불만족', '상품', '상품'}

# 제품별 통계 저장
product_stats = {}

# 제품별로 그룹화
product_groups = analysis_df.groupby('product_id')

# 각 제품별 통계 계산
for product_id, group in product_groups:
    product_name = group.iloc[0]['product_name']
    
    # 기본 통계
    stats = {
        'name': product_name,
        'review_count': len(group),
        'avg_rating': group['rating'].mean(),
        'rating_std': group['rating'].std(),
        'avg_recommendation': group['recommendation_score'].mean(),
        'sentiment_distribution': group['overall_sentiment'].value_counts().to_dict()
    }
    
    # 측면별 감정 분석 데이터 수집 (긍정/부정 구분)
    aspect_sentiment = defaultdict(list)
    aspect_positive = defaultdict(int)  # 측면별 긍정 개수
    aspect_negative = defaultdict(int)  # 측면별 부정 개수
    all_pros = []
    all_cons = []
    
    for _, review in group.iterrows():
        # 장단점 수집 (불용어 제거)
        if isinstance(review['pros'], list):
            filtered_pros = [pro for pro in review['pros'] if pro.strip() not in stopwords]
            all_pros.extend(filtered_pros)
        if isinstance(review['cons'], list):
            filtered_cons = [con for con in review['cons'] if con.strip() not in stopwords]
            all_cons.extend(filtered_cons)
        
        # 측면별 감정 수집
        if isinstance(review['aspect_analyses'], list):
            for aspect_analysis in review['aspect_analyses']:
                if isinstance(aspect_analysis, dict):
                    aspect = aspect_analysis.get('aspect')
                    sentiment = aspect_analysis.get('sentiment')
                    if aspect and sentiment:
                        aspect_sentiment[aspect].append(sentiment)
                        # 긍정/부정 카운트
                        if sentiment in ['positive', 'very_positive']:
                            aspect_positive[aspect] += 1
                        elif sentiment in ['negative', 'very_negative']:
                            aspect_negative[aspect] += 1
    
    # 측면별 긍정 비율 계산
    aspect_scores = {}
    for aspect, sentiments in aspect_sentiment.items():
        positive_count = sum(1 for s in sentiments if s in ['positive', 'very_positive'])
        if len(sentiments) > 0:
            aspect_scores[aspect] = positive_count / len(sentiments)
    
    # 통계에 추가 정보 저장
    stats['aspect_scores'] = aspect_scores
    stats['aspect_positive'] = aspect_positive
    stats['aspect_negative'] = aspect_negative
    stats['top_pros'] = [item for item, count in Counter(all_pros).most_common(5)]
    stats['top_cons'] = [item for item, count in Counter(all_cons).most_common(5)]
    stats['all_pros'] = all_pros  # 워드클라우드용
    stats['all_cons'] = all_cons  # 워드클라우드용
    
    product_stats[product_id] = stats

# =============================================================================
# 비교 결과 요약 (텍스트로 저장)
# =============================================================================

# 측면 한글 매핑
aspect_korean = {
    'quality': '품질',
    'usability': '사용성',
    'functionality': '기능성',
    'design': '디자인',
    'price_value': '가성비',
    'durability': '내구성',
    'customer_service': '고객서비스',
    'shipping': '배송',
    'packaging': '포장',
    'overall': '전반적'
}

# 요약 텍스트를 변수에 저장
summary_text = []
summary_text.append("제품 비교 요약:")
summary_text.append("=" * 60)

for product_id, stats in product_stats.items():
    summary_text.append(f"\n제품: {stats['name'][:40]}")
    summary_text.append(f"  리뷰 수: {stats['review_count']}개")
    summary_text.append(f"  평균 평점: {stats['avg_rating']:.2f}점")
    summary_text.append(f"  평균 추천도: {stats['avg_recommendation']:.2f}")
    summary_text.append(f"  주요 장점: {', '.join(stats['top_pros'][:5])}")
    summary_text.append(f"  주요 단점: {', '.join(stats['top_cons'][:5])}")
    
    # 측면별 만족도 출력
    if stats['aspect_scores']:
        summary_text.append(f"  측면별 만족도 (긍정 비율):")
        # 만족도 높은 순으로 정렬
        sorted_aspects = sorted(stats['aspect_scores'].items(), key=lambda x: x[1], reverse=True)
        for aspect, score in sorted_aspects:
            aspect_kr = aspect_korean.get(aspect, aspect)
            summary_text.append(f"    - {aspect_kr}: {score:.2f} ({score*100:.1f}%)")

# 리스트를 문자열로 결합
product_comparison_summary = "\n".join(summary_text)

# 콘솔에 출력
print(product_comparison_summary)

# 텍스트 파일로 저장
summary_txt_path = f"{output_dir}/product_comparison_summary.txt"
with open(summary_txt_path, 'w', encoding='utf-8') as f:
    f.write(product_comparison_summary)

In [ ]:
# 제품 하나에 대한 종합 결과
product_stats[6842777459]

In [ ]:
# =============================================================================
# 시각화
# =============================================================================
from collections import Counter
import plotly.graph_objects as go
import plotly.express as px
from wordcloud import WordCloud
import matplotlib.pyplot as plt

print("제품 비교 시각화 생성 중...\n")

# 기본 데이터 준비
products = list(product_stats.keys())
product_names = [product_stats[pid]['name'][:20] + '...' for pid in products]

# 1. 평균 평점 및 추천도 비교
print("1. 평균 평점 및 추천도 비교")
avg_ratings = [product_stats[pid]['avg_rating'] for pid in products]
rating_stds = [product_stats[pid]['rating_std'] for pid in products]
recommendations = [product_stats[pid]['avg_recommendation'] for pid in products]

fig1 = go.Figure()

# 평균 평점 (왼쪽 Y축)
fig1.add_trace(go.Bar(
    name='평균 평점',
    x=product_names,
    y=avg_ratings,
    error_y=dict(type='data', array=rating_stds),
    marker_color='lightblue',
    text=[f'{rating:.2f}' for rating in avg_ratings],
    textposition='auto',
    yaxis='y'
))

# 추천도 (오른쪽 Y축)
fig1.add_trace(go.Scatter(
    name='추천도',
    x=product_names,
    y=recommendations,
    mode='lines+markers',
    marker=dict(size=10, color='orange'),
    line=dict(width=3, color='orange'),
    text=[f'{rec:.2f}' for rec in recommendations],
    textposition='top center',
    yaxis='y2'
))

# Y축 최대값 계산 (에러바 고려)
max_rating_with_error = max([rating + std for rating, std in zip(avg_ratings, rating_stds)])
y_axis_max = min(max_rating_with_error * 1.3, 6.0)

fig1.update_layout(
    title='제품별 평균 평점 및 추천도 비교',
    xaxis_title='제품명',
    yaxis=dict(
        title='평점 (1-5점)',
        side='left',
        range=[0, y_axis_max]
    ),
    yaxis2=dict(
        title='추천도 (0-1)',
        side='right',
        overlaying='y',
        range=[0, 1.1]
    ),
    height=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig1.show()
fig1.write_image(f"{output_dir}/rating_recommendation_comparison.png", width=1200, height=700, scale=2)

In [ ]:
# 2. 제품별 측면별 긍정/부정 비율 (스택 막대)
print("2. 제품별 측면별 긍정/부정 비율")

# 측면 한글 매핑
aspect_korean = {
    'quality': '품질',
    'usability': '사용성',
    'functionality': '기능성',
    'design': '디자인',
    'price_value': '가성비',
    'durability': '내구성',
    'customer_service': '고객서비스',
    'shipping': '배송',
    'packaging': '포장',
    'overall': '전반적'
}

for product_id, stats in product_stats.items():
    product_name = stats['name'][:40]
    
    # 해당 제품의 측면들
    aspects = sorted(set(list(stats['aspect_positive'].keys()) + list(stats['aspect_negative'].keys())))
    
    if aspects:
        # 긍정/부정 개수
        positive_counts = [stats['aspect_positive'].get(aspect, 0) for aspect in aspects]
        negative_counts = [stats['aspect_negative'].get(aspect, 0) for aspect in aspects]
        
        # 만족도(긍정 비율) 계산
        satisfaction_ratios = []
        for aspect in aspects:
            pos = stats['aspect_positive'].get(aspect, 0)
            neg = stats['aspect_negative'].get(aspect, 0)
            total = pos + neg
            ratio = (pos / total * 100) if total > 0 else 0
            satisfaction_ratios.append(f"{ratio:.1f}%")
        
        # 한글 측면명
        aspects_kr = [aspect_korean.get(aspect, aspect) for aspect in aspects]
        
        fig2 = go.Figure()
        
        # 긍정 막대 (아래)
        fig2.add_trace(go.Bar(
            name='긍정',
            x=aspects_kr,
            y=positive_counts,
            marker_color='lightgreen',
            text=positive_counts,
            textposition='inside'
        ))
        
        # 부정 막대 (위에 스택)
        fig2.add_trace(go.Bar(
            name='부정',
            x=aspects_kr,
            y=negative_counts,
            marker_color='lightcoral',
            text=negative_counts,
            textposition='inside'
        ))
        
        # 만족도 텍스트 추가 (그래프 상단)
        total_counts = [p + n for p, n in zip(positive_counts, negative_counts)]
        fig2.add_trace(go.Scatter(
            x=aspects_kr,
            y=[total + max(total_counts) * 0.05 for total in total_counts],  # 막대 위에 표시
            mode='text',
            text=[f'만족도: {ratio}' for ratio in satisfaction_ratios],
            textposition='top center',
            textfont=dict(size=11, color='black', family='Arial Black'),
            showlegend=False
        ))
        
        fig2.update_layout(
            title=f'{product_name} - 측면별 긍정/부정 비교',
            xaxis_title='분석 측면',
            yaxis_title='리뷰 수',
            barmode='stack',
            height=500,
            showlegend=True
        )
        fig2.show()
        fig2.write_image(f"{output_dir}/aspect_analysis_product_{product_id}.png", width=1200, height=500, scale=2)

In [ ]:
# 3. 제품별 측면-감정 트리맵
print("3. 제품별 측면-감정 트리맵")
print("""트리맵 해석: 사각형의 크기는 리뷰 개수, 색상은 감정(초록=긍정, 노랑=중립, 빨강=부정)을 나타냅니다.
각 측면(품질, 디자인 등) 안의 긍정/부정/중립 비율을 한눈에 비교할 수 있습니다.
""")

for product_id, stats in product_stats.items():
    product_name = stats['name'][:30]
    
    # 측면 한글 매핑
    aspect_korean = {
        'quality': '품질',
        'usability': '사용성',
        'functionality': '기능성',
        'design': '디자인',
        'price_value': '가성비',
        'durability': '내구성',
        'customer_service': '고객서비스',
        'shipping': '배송',
        'packaging': '포장',
        'overall': '전반적'
    }
    
    # 트리맵용 데이터 준비
    labels = []
    parents = []
    values = []
    colors = []
    
    # 루트
    root_name = product_name
    labels.append(root_name)
    parents.append("")
    values.append(0)
    colors.append(0)
    
    # 측면별 데이터 수집 (긍정/중립/부정으로 통합)
    aspect_data = defaultdict(lambda: {'긍정': 0, '중립': 0, '부정': 0})
    
    for _, review in analysis_df[analysis_df['product_id'] == product_id].iterrows():
        if isinstance(review['aspect_analyses'], list):
            for aspect_analysis in review['aspect_analyses']:
                if isinstance(aspect_analysis, dict):
                    aspect = aspect_analysis.get('aspect')
                    sentiment = aspect_analysis.get('sentiment')
                    if aspect and sentiment:
                        # 감정을 3가지로 통합
                        if sentiment in ['positive', 'very_positive']:
                            aspect_data[aspect]['긍정'] += 1
                        elif sentiment in ['negative', 'very_negative']:
                            aspect_data[aspect]['부정'] += 1
                        else:
                            aspect_data[aspect]['중립'] += 1
    
    # 감정 색상 매핑
    sentiment_colors = {
        '긍정': 1,
        '중립': 0,
        '부정': -1
    }
    
    # 측면별 노드 추가
    for aspect, sentiments in aspect_data.items():
        aspect_total = sum(sentiments.values())
        if aspect_total == 0:
            continue
            
        # 측면의 평균 감정 점수 계산
        aspect_avg_color = sum(sentiment_colors.get(s, 0) * count 
                               for s, count in sentiments.items()) / aspect_total
        
        # 측면명을 한글로 변환
        aspect_kr = aspect_korean.get(aspect, aspect)
        
        labels.append(aspect_kr)
        parents.append(root_name)
        values.append(aspect_total)
        colors.append(aspect_avg_color)
        
        # 감정별 세부 노드 (측면-감정 형태)
        for sentiment, count in sentiments.items():
            if count > 0:
                labels.append(f"{aspect_kr}-{sentiment}")  # 한글 측면-감정
                parents.append(aspect_kr)
                values.append(count)
                colors.append(sentiment_colors.get(sentiment, 0))
    
    # 트리맵 생성
    if len(labels) > 1:
        fig = px.treemap(
            names=labels,
            parents=parents,
            values=values,
            color=colors,
            color_continuous_scale='RdYlGn',
            color_continuous_midpoint=0,
            title=f'{product_name} - 측면별 감정 분포 (트리맵)'
        )
        
        fig.update_traces(
            textposition='middle center',
            textfont=dict(size=12),
            hovertemplate='<b>%{label}</b><br>리뷰 수: %{value}<br><extra></extra>'
        )
        
        fig.update_layout(
            height=600,
            margin=dict(t=50, l=25, r=25, b=25)
        )
        fig.show()
        fig.write_image(f"{output_dir}/aspect_sentiment_treemap_{product_id}.png", width=1200, height=600, scale=2)

In [ ]:
# 4. 측면별 긍정 비율 히트맵
print("4. 측면별 긍정 비율 히트맵")

# 모든 측면 수집
all_aspects = set()
for stats in product_stats.values():
    all_aspects.update(stats['aspect_scores'].keys())
all_aspects = sorted(list(all_aspects))

# 히트맵 데이터 준비
heatmap_data = []
for pid in products:
    row = [product_stats[pid]['aspect_scores'].get(aspect, 0) for aspect in all_aspects]
    heatmap_data.append(row)

fig4 = go.Figure()
fig4.add_trace(go.Heatmap(
    z=heatmap_data,
    x=all_aspects,
    y=product_names,
    colorscale='RdYlGn',
    text=[[f'{val:.2f}' for val in row] for row in heatmap_data],
    texttemplate='%{text}',
    textfont={"size": 10},
    colorbar=dict(title="긍정 비율")
))
fig4.update_layout(
    title='제품별 측면별 긍정 비율 히트맵',
    xaxis_title='분석 측면',
    yaxis_title='제품명',
    height=400 + len(products) * 50
)
fig4.show()
fig4.write_image(f"{output_dir}/aspect_positive_ratio_heatmap.png", width=1200, height=400 + len(products) * 50, scale=2)

In [ ]:
# 5. 감정 분포 비교
print("5. 감정 분포 비교")

# 감정 순서 정의 (부정 → 긍정)
sentiment_order = ['very_negative', 'negative', 'neutral', 'positive', 'very_positive']

# 실제 데이터에 있는 감정만 필터링
all_sentiments = set()
for stats in product_stats.values():
    all_sentiments.update(stats['sentiment_distribution'].keys())

# 정의된 순서에 따라 정렬
ordered_sentiments = [s for s in sentiment_order if s in all_sentiments]

fig5 = go.Figure()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# 제품별 감정 분포
for i, (product_id, stats) in enumerate(product_stats.items()):
    sentiment_dist = stats['sentiment_distribution']
    counts = [sentiment_dist.get(sentiment, 0) for sentiment in ordered_sentiments]
    
    fig5.add_trace(go.Bar(
        name=product_names[i],
        x=ordered_sentiments,
        y=counts,
        marker_color=colors[i % len(colors)],
        text=counts,
        textposition='auto'
    ))

fig5.update_layout(
    title='제품별 감정 분포 비교',
    xaxis_title='감정 유형 (부정 → 긍정)',
    yaxis_title='리뷰 수',
    barmode='group',
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig5.show()
fig5.write_image(f"{output_dir}/sentiment_distribution_comparison.png", width=1200, height=500, scale=2)

In [ ]:
# 6. 제품별 장점/단점 워드클라우드
print("6. 제품별 장점/단점 워드클라우드")

import platform
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한글 폰트 설정
if platform.system() == 'Windows':
    korean_font = 'C:/Windows/Fonts/malgun.ttf'
    plt.rc('font', family='Malgun Gothic')
else:  # macOS
    korean_font = '/System/Library/Fonts/AppleSDGothicNeo.ttc'
    plt.rc('font', family='AppleGothic')

# 마이너스 기호 깨짐 방지
plt.rc('axes', unicode_minus=False)

# 워드클라우드용 불용어
wordcloud_stopwords = {
    '없음', '있음', '사용', '좋음', '나쁨', '보통', '괜찮음',
    '추천', '적극추천', '만족', '불만족', '상품', '상품만족',
    '가능', '않음', '매우', '정말', '너무', '아주', '완전', '진짜', '것', '거', '이'
}

# 제품별로 워드클라우드 생성
for product_id, stats in product_stats.items():
    product_name = stats['name'][:30]
    
    # 서브플롯 생성 (장점/단점 나란히)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'{product_name} - 장점/단점 워드클라우드', fontsize=16, fontweight='bold')
    
    # 장점 워드클라우드
    if stats['all_pros']:
        pros_text = ' '.join(stats['all_pros'])
        wordcloud_pros = WordCloud(
            width=800, 
            height=400,
            background_color='white',
            colormap='Greens',
            font_path=korean_font,
            relative_scaling=0.5,
            min_font_size=10,
            stopwords=wordcloud_stopwords
        ).generate(pros_text)
        
        axes[0].imshow(wordcloud_pros, interpolation='bilinear')
        axes[0].set_title('장점', fontsize=14, color='green')
        axes[0].axis('off')
    else:
        axes[0].text(0.5, 0.5, '장점 데이터 없음', ha='center', va='center')
        axes[0].axis('off')
    
    # 단점 워드클라우드
    if stats['all_cons']:
        cons_text = ' '.join(stats['all_cons'])
        wordcloud_cons = WordCloud(
            width=800,
            height=400,
            background_color='white',
            colormap='Reds',
            font_path=korean_font,
            relative_scaling=0.5,
            min_font_size=10,
            stopwords=wordcloud_stopwords
        ).generate(cons_text)
        
        axes[1].imshow(wordcloud_cons, interpolation='bilinear')
        axes[1].set_title('단점', fontsize=14, color='red')
        axes[1].axis('off')
    else:
        axes[1].text(0.5, 0.5, '단점 데이터 없음', ha='center', va='center')
        axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/pros_cons_wordcloud_{product_id}.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# 7. 제품별 측면별 긍정/부정 키워드 워드클라우드
print("7. 제품별 측면별 긍정/부정 키워드 워드클라우드")

from collections import defaultdict

# 워드클라우드용 불용어
wordcloud_stopwords = {
    '없음', '있음', '사용', '좋음', '나쁨', '보통', '괜찮음',
    '추천', '적극추천', '만족', '불만족', '상품', '상품만족',
    '가능', '않음', '매우', '정말', '너무', '아주', '완전', '진짜', '것', '거', '이'
}


# 제품별로 측면별 키워드 수집
for product_id, stats in product_stats.items():
    product_name = stats['name'][:30]
    
    # 측면별 긍정/부정 키워드 수집
    aspect_positive_keywords = defaultdict(list)
    aspect_negative_keywords = defaultdict(list)
    
    for _, review in analysis_df[analysis_df['product_id'] == product_id].iterrows():
        if isinstance(review.get('aspect_analyses'), list):
            for aspect in review['aspect_analyses']:
                if isinstance(aspect, dict):
                    aspect_name = aspect.get('aspect', '')
                    aspect_sentiment = aspect.get('sentiment', '')
                    keywords = aspect.get('keywords', [])
                    
                    if keywords:
                        # 긍정적 감정
                        if aspect_sentiment in ['positive', 'very_positive']:
                            aspect_positive_keywords[aspect_name].extend(keywords)
                        # 부정적 감정
                        elif aspect_sentiment in ['negative', 'very_negative']:
                            aspect_negative_keywords[aspect_name].extend(keywords)
    
    # 측면 한글 매핑
    aspect_korean = {
        'quality': '품질',
        'usability': '사용성',
        'functionality': '기능성',
        'design': '디자인',
        'price_value': '가성비',
        'durability': '내구성',
        'customer_service': '고객서비스',
        'shipping': '배송',
        'packaging': '포장',
        'overall': '전반적'
    }
    
    # 긍정 키워드가 있는 측면들
    positive_aspects = [aspect for aspect in aspect_positive_keywords.keys() if aspect_positive_keywords[aspect]]
    
    if positive_aspects:
        # 긍정 키워드 워드클라우드
        n_aspects = len(positive_aspects)
        n_cols = min(3, n_aspects)
        n_rows = (n_aspects + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
        fig.suptitle(f'{product_name} - 측면별 긍정 키워드', fontsize=16, fontweight='bold')
        
        if n_aspects == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        for idx, aspect in enumerate(positive_aspects):
            keywords_text = ' '.join(aspect_positive_keywords[aspect])
            aspect_kr = aspect_korean.get(aspect, aspect)
            
            if keywords_text:
                wordcloud = WordCloud(
                    width=600,
                    height=400,
                    background_color='white',
                    colormap='Greens',
                    font_path=korean_font,
                    relative_scaling=0.5,
                    min_font_size=10,
                    stopwords=wordcloud_stopwords
                ).generate(keywords_text)
                
                axes[idx].imshow(wordcloud, interpolation='bilinear')
                axes[idx].set_title(f'{aspect_kr} (긍정)', fontsize=12, color='green', fontweight='bold')
                axes[idx].axis('off')
        
        # 빈 서브플롯 제거
        for idx in range(len(positive_aspects), len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/aspect_positive_keywords_wordcloud_{product_id}.png", dpi=300, bbox_inches='tight')
        plt.show()
    
    # 부정 키워드가 있는 측면들
    negative_aspects = [aspect for aspect in aspect_negative_keywords.keys() if aspect_negative_keywords[aspect]]
    
    if negative_aspects:
        # 부정 키워드 워드클라우드
        n_aspects = len(negative_aspects)
        n_cols = min(3, n_aspects)
        n_rows = (n_aspects + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
        fig.suptitle(f'{product_name} - 측면별 부정 키워드', fontsize=16, fontweight='bold')
        
        if n_aspects == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        for idx, aspect in enumerate(negative_aspects):
            keywords_text = ' '.join(aspect_negative_keywords[aspect])
            aspect_kr = aspect_korean.get(aspect, aspect)
            
            if keywords_text:
                wordcloud = WordCloud(
                    width=600,
                    height=400,
                    background_color='white',
                    colormap='Reds',
                    font_path=korean_font,
                    relative_scaling=0.5,
                    min_font_size=10,
                    stopwords=wordcloud_stopwords
                ).generate(keywords_text)
                
                axes[idx].imshow(wordcloud, interpolation='bilinear')
                axes[idx].set_title(f'{aspect_kr} (부정)', fontsize=12, color='red', fontweight='bold')
                axes[idx].axis('off')
        
        # 빈 서브플롯 제거
        for idx in range(len(negative_aspects), len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/aspect_negative_keywords_wordcloud_{product_id}.png", dpi=300, bbox_inches='tight')
        plt.show()